# CLARITY Generalized API Pipeline Experiment

This notebook runs the experimental CLARITY prompt pipeline for any de-identified clinician-written medical note. It reads a Word `.docx` note, extracts a shared clinical fact base, creates a shared patient-facing explanation plan, generates five script versions, and optionally audits each script for medical faithfulness and visual safety.

The notebook is intentionally separate from the Streamlit dashboard so prompt/API experiments can be audited before they are promoted into the demo UI.

> Safety note
>
> Do not send protected health information to external APIs unless your workflow, storage, vendor agreement, and review process are approved for that use. This experiment is intended for de-identified interview/demo materials. Generated scripts require clinician review before patient-facing use.

In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI, OpenAIError


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook launched in any folder.

    Args:
        start: Optional starting directory. The current working directory is used
            when no value is provided.

    Returns:
        Repository root containing `pyproject.toml`, `src/`, and `prompts/`.

    CLARITY pipeline role:
        Keeps the experiment portable across VS Code, JupyterLab, and terminal
        launch contexts while still reusing the prototype's Word parser and
        prompt files.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the CLARITY repo root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.clarity_dashboard.io_utils import extract_docx_text

PROMPT_DIR = REPO_ROOT / "prompts"
NOTEBOOK_DIR = REPO_ROOT / "experiments" / "clarity_api_pipeline"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(REPO_ROOT / ".env")

MODEL = os.getenv("LLM_MODEL", "gpt-4.1-mini")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print(f"Repo root: {REPO_ROOT}")
print(f"Prompt dir: {PROMPT_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Model: {MODEL}")
print(f"OpenAI key configured: {bool(os.getenv('OPENAI_API_KEY'))}")


In [ ]:
# Update this before running live API calls.
# The input should be a de-identified Word document.
DOCX_PATH = REPO_ROOT / "data" / "your_deidentified_clinical_note.docx"

# Keep this False until the path and key are correct. This prevents accidental
# API calls while opening or testing the notebook.
RUN_LIVE_API = False

VERSIONS_TO_GENERATE = ["control", "spanish", "trust", "technical", "simple_vague"]
RUN_AUDIT_AFTER_SCRIPT = True


In [ ]:
VERSION_PROMPTS = {
    "control": "03A_control_user.txt",
    "spanish": "03B_spanish_user.txt",
    "trust": "03C_trust_user.txt",
    "technical": "03D_technical_user.txt",
    "simple_vague": "03E_simple_vague_user.txt",
}


def load_prompt(filename: str) -> str:
    """Load one prompt asset from the repository `prompts/` directory.

    Args:
        filename: Prompt filename, such as `01_extract_facts_system.txt`.

    Returns:
        UTF-8 prompt text.

    CLARITY pipeline role:
        Makes the notebook use the same editable prompt files that future
        Streamlit/API integration can reuse, instead of burying prompts inside
        notebook outputs.
    """
    return (PROMPT_DIR / filename).read_text(encoding="utf-8")


def render_template(template: str, **kwargs: Any) -> str:
    """Render prompt placeholders such as `{{FACT_BASE_JSON}}`.

    Args:
        template: Prompt text containing double-brace placeholders.
        **kwargs: Replacement values. Dictionaries are serialized as JSON.

    Returns:
        Prompt text ready to send to the model.

    CLARITY pipeline role:
        Preserves the core design principle: facts and explanation plans are
        shared, while version-specific prompts only control expression style.
    """
    rendered = template
    for key, value in kwargs.items():
        if not isinstance(value, str):
            value = json.dumps(value, ensure_ascii=False, indent=2)
        rendered = rendered.replace("{{" + key + "}}", value)
    return rendered


def load_docx_note(path: Path) -> str:
    """Extract plain text from a de-identified Word clinical note.

    Args:
        path: Path to a `.docx` clinical note.

    Returns:
        Plain text extracted from the Word document.

    CLARITY pipeline role:
        Implements Step 1 for the API experiment. Word parsing is kept separate
        from LLM-based fact extraction so the model receives normalized note
        text rather than document-format details.
    """
    if not path.exists():
        raise FileNotFoundError(f"DOCX_PATH does not exist: {path}")
    if path.suffix.lower() != ".docx":
        raise ValueError("This experiment expects a .docx file. Convert other formats first or update this loader.")
    text = extract_docx_text(path.read_bytes())
    if not text.strip():
        raise ValueError("No readable text was extracted. If this is a scanned document, run OCR first.")
    return text


In [ ]:
def parse_json_object(text: str) -> dict:
    """Parse a model response expected to contain one JSON object.

    Args:
        text: Raw model output.

    Returns:
        Parsed JSON object.

    CLARITY pipeline role:
        Keeps every API stage machine-readable. The parser tolerates accidental
        markdown fences during prompt iteration, but still requires a top-level
        object.
    """
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start : end + 1]
    parsed = json.loads(cleaned)
    if not isinstance(parsed, dict):
        raise TypeError("Expected a top-level JSON object.")
    return parsed


def extract_response_text(response: Any) -> str:
    """Extract text from an OpenAI Responses API object.

    Args:
        response: Object returned by `client.responses.create`.

    Returns:
        Best-effort response text.

    CLARITY pipeline role:
        Provides a small compatibility layer so downstream JSON parsing does
        not depend on a single SDK response shape.
    """
    output_text = getattr(response, "output_text", "")
    if output_text:
        return output_text
    chunks = []
    for item in getattr(response, "output", []) or []:
        for content in getattr(item, "content", []) or []:
            text = getattr(content, "text", "")
            if text:
                chunks.append(text)
    return "\n".join(chunks).strip()


def call_model_json(client: OpenAI, model: str, system_prompt: str, user_prompt: str) -> dict:
    """Call the OpenAI Responses API and return a parsed JSON object.

    Args:
        client: Configured OpenAI client.
        model: Model name from `.env`, for example `gpt-4.1-mini`.
        system_prompt: Shared or stage-specific high-priority instructions.
        user_prompt: Case-specific prompt content.

    Returns:
        Parsed JSON object from the model. If the first response is invalid
        JSON, the function retries once with a repair instruction.

    CLARITY pipeline role:
        Centralizes the API behavior for extraction, planning, script
        generation, and audit. JSON mode plus a one-time repair retry makes
        the notebook practical for prompt experiments without hiding failures.
    """
    try:
        response = client.responses.create(
            model=model,
            input=[
                {"role": "developer", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            text={"format": {"type": "json_object"}},
        )
        raw_text = extract_response_text(response)
        return parse_json_object(raw_text)
    except (json.JSONDecodeError, TypeError) as parse_error:
        repair_prompt = (
            "The previous response could not be parsed as valid JSON. "
            "Return only one valid JSON object that preserves the same content.\n\n"
            f"Previous response:\n{raw_text}"
        )
        repair_response = client.responses.create(
            model=model,
            input=[
                {"role": "developer", "content": "Return valid JSON only."},
                {"role": "user", "content": repair_prompt},
            ],
            text={"format": {"type": "json_object"}},
        )
        return parse_json_object(extract_response_text(repair_response))
    except OpenAIError:
        raise


In [ ]:
def save_json(data: dict, filename: str) -> Path:
    """Save one generated pipeline artifact to the ignored outputs folder."""
    path = OUTPUT_DIR / filename
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def extract_facts(client: OpenAI, model: str, clinical_note: str) -> dict:
    """Run Stage 1: clinical note to shared structured fact base."""
    return call_model_json(
        client=client,
        model=model,
        system_prompt=load_prompt("01_extract_facts_system.txt"),
        user_prompt=render_template(load_prompt("01_extract_facts_user.txt"), CLINICAL_NOTE=clinical_note),
    )


def create_explanation_plan(client: OpenAI, model: str, fact_base: dict) -> dict:
    """Run Stage 2: shared fact base to shared patient-facing explanation plan."""
    return call_model_json(
        client=client,
        model=model,
        system_prompt=load_prompt("02_explanation_plan_system.txt"),
        user_prompt=render_template(load_prompt("02_explanation_plan_user.txt"), FACT_BASE_JSON=fact_base),
    )


def generate_script(client: OpenAI, model: str, version: str, fact_base: dict, explanation_plan: dict) -> dict:
    """Run Stage 3 for one version using shared facts and version-specific style."""
    return call_model_json(
        client=client,
        model=model,
        system_prompt=load_prompt("03_shared_script_system.txt"),
        user_prompt=render_template(
            load_prompt(VERSION_PROMPTS[version]),
            FACT_BASE_JSON=fact_base,
            EXPLANATION_PLAN_JSON=explanation_plan,
        ),
    )


def audit_script(client: OpenAI, model: str, version: str, fact_base: dict, explanation_plan: dict, script: dict) -> dict:
    """Run Stage 4: audit one generated script for medical and visual safety."""
    version_requirements = load_prompt(VERSION_PROMPTS[version])
    return call_model_json(
        client=client,
        model=model,
        system_prompt=load_prompt("04_visual_and_medical_audit_system.txt"),
        user_prompt=render_template(
            load_prompt("04_visual_and_medical_audit_user.txt"),
            FACT_BASE_JSON=fact_base,
            EXPLANATION_PLAN_JSON=explanation_plan,
            VERSION_REQUIREMENTS=version_requirements,
            SCRIPT_JSON=script,
        ),
    )


In [ ]:
if not RUN_LIVE_API:
    print("RUN_LIVE_API is False. Set it to True after configuring DOCX_PATH and OPENAI_API_KEY.")
else:
    clinical_note = load_docx_note(DOCX_PATH)
    print(f"Loaded note characters: {len(clinical_note):,}")

    fact_base = extract_facts(client, MODEL, clinical_note)
    save_json(fact_base, "01_fact_base.json")

    explanation_plan = create_explanation_plan(client, MODEL, fact_base)
    save_json(explanation_plan, "02_explanation_plan.json")

    scripts = {}
    audits = {}
    for version in VERSIONS_TO_GENERATE:
        scripts[version] = generate_script(client, MODEL, version, fact_base, explanation_plan)
        save_json(scripts[version], f"03_{version}_script.json")

        if RUN_AUDIT_AFTER_SCRIPT:
            audits[version] = audit_script(client, MODEL, version, fact_base, explanation_plan, scripts[version])
            save_json(audits[version], f"04_{version}_audit.json")

    summary = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "model": MODEL,
        "docx_path": str(DOCX_PATH),
        "versions_generated": VERSIONS_TO_GENERATE,
        "audit_enabled": RUN_AUDIT_AFTER_SCRIPT,
        "outputs": sorted(path.name for path in OUTPUT_DIR.glob("*.json")),
    }
    save_json(summary, "00_run_summary.json")
    summary


## How To Use The Outputs

- `01_fact_base.json` is the shared clinical source of truth for the case.
- `02_explanation_plan.json` is the shared patient-facing structure all versions should follow.
- `03_*_script.json` files are the five style-controlled script outputs.
- `04_*_audit.json` files flag unsupported claims, missing facts, uncertainty problems, and prohibited clinician/doctor visual suggestions.

These outputs are written under `experiments/clarity_api_pipeline/outputs/`, which is ignored by git because generated clinical text should be reviewed before sharing.